# Notebook 2: Data Cleaning, Pre-Processing & Dimensionality Reduction
---

### Key fix in this version:
 This version runs outlier removal **within each genre separately** so that hip hop's legitimate signal is preserved rather than treated as noise.

### Steps:
1. Drop identity and irrelevant columns
2. Drop low-variance features identified in EDA
3. Handle missing values
4. Remove outliers per genre using IQR
5. Scale features using StandardScaler
6. Save cleaned dataset and scaler

---
## Setup: Imports and Data Loading

Import all required libraries and load the raw training data. We confirm the shape matches what we saw in EDA (28,362 rows x 24 columns) before making any changes.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.preprocessing import StandardScaler

%matplotlib inline
sns.set_style('whitegrid')

# File paths
BASE_PATH      = '/Users/saadult/Music Rec Algo/Music-Recommendation-Algorithm/data/'
TRAIN_RAW_PATH = BASE_PATH + 'train.csv'

# Load raw training data
df = pd.read_csv(TRAIN_RAW_PATH)

print(f'Raw data shape: {df.shape}')
print(f'Genres in dataset: {df["genre"].unique().tolist()}')
print(f'Songs per genre:')
print(df['genre'].value_counts())
df.head()

Raw data shape: (28362, 24)
Genres in dataset: ['pop', 'country', 'blues', 'jazz', 'reggae', 'rock', 'hip hop']
Songs per genre:
genre
pop        7038
country    5444
blues      4603
rock       4032
jazz       3844
reggae     2497
hip hop     904
Name: count, dtype: int64


,Unnamed: 0,artist_name,track_name,release_date,genre,lyrics,len,dating,violence,world/life,...,communication,obscene,music,movement/places,light/visual perceptions,family/spiritual,sadness,feelings,topic,age
0,0,mukesh,mohabbat bhi jhoothi,1950,pop,hold time feel break feel untrue convince spea...,95,0.000598,0.063746,0.000598,...,0.263751,0.000598,0.039288,0.000598,0.000598,0.000598,0.380299,0.117175,sadness,1.0
1,4,frankie laine,i believe,1950,pop,believe drop rain fall grow believe darkest ni...,51,0.035537,0.096777,0.443435,...,0.001284,0.001284,0.118034,0.001284,0.212681,0.051124,0.001284,0.001284,world/life,1.0
2,6,johnnie ray,cry,1950,pop,sweetheart send letter goodbye secret feel bet...,24,0.002770,0.002770,0.002770,...,0.250668,0.002770,0.323794,0.002770,0.002770,0.002770,0.002770,0.225422,music,1.0
3,10,pérez prado,patricia,1950,pop,kiss lips want stroll charm mambo chacha merin...,54,0.048249,0.001548,0.001548,...,0.001548,0.001548,0.001548,0.129250,0.001548,0.001548,0.225889,0.001548,romantic,1.0
4,12,giorgos papadopoulos,apopse eida oneiro,1950,pop,till darling till matter know till dream live ...,48,0.001350,0.001350,0.417772,...,0.001350,0.001350,0.001350,0.001350,0.001350,0.029755,0.068800,0.001350,romantic,1.0


---
## Step 1: Drop Identity and Irrelevant Columns

These columns are labels or metadata — they cannot be used by KMeans because they are either text or index values with no numeric meaning. The `lyrics` column is dropped per the assignment instructions; it requires a Word2Vec model (optional bonus challenge) to be useful.

In [2]:
# Drop identity and non-numeric columns
# 'Unnamed: 0'  — row index from original CSV, carries no information
# 'lyrics'      — raw text, requires NLP/Word2Vec to use
# 'artist_name' — identity label, not a lyrical feature
# 'track_name'  — identity label, not a lyrical feature

cols_to_drop_identity = ['Unnamed: 0', 'lyrics', 'artist_name', 'track_name']
df_clean = df.drop(
    columns=[c for c in cols_to_drop_identity if c in df.columns]
)

print(f'Shape after dropping identity columns: {df_clean.shape}')
print(f'Remaining columns: {df_clean.columns.tolist()}')

Shape after dropping identity columns: (28362, 20)
Remaining columns: ['release_date', 'genre', 'len', 'dating', 'violence', 'world/life', 'night/time', 'shake the audience', 'family/gospel', 'romantic', 'communication', 'obscene', 'music', 'movement/places', 'light/visual perceptions', 'family/spiritual', 'sadness', 'feelings', 'topic', 'age']


---
## Step 2: Drop Low-Variance Features

**EDA justification:** The variance analysis in Notebook 1 identified four score columns with variance below 0.003. These columns are so heavily skewed toward zero that most songs look nearly identical on these dimensions, giving the algorithm almost no ability to separate songs using them.

Columns being dropped and their variances:
- `shake the audience` — 0.00165
- `family/gospel` — 0.00176
- `family/spiritual` — 0.00260
- `dating` — 0.00274

No columns are dropped for high correlation because EDA showed no pairs exceeded 0.27 — well below the 0.7 redundancy threshold.

In [3]:
# Drop the four lowest-variance score columns identified in EDA
# These features lack sufficient spread to contribute meaningful cluster separation
low_variance_cols = [
    'shake the audience',  # variance: 0.00165
    'family/gospel',       # variance: 0.00176
    'family/spiritual',    # variance: 0.00260
    'dating',              # variance: 0.00274
]

df_clean = df_clean.drop(
    columns=[c for c in low_variance_cols if c in df_clean.columns]
)

print(f'Shape after dropping low-variance columns: {df_clean.shape}')
print(f'Remaining columns: {df_clean.columns.tolist()}')

Shape after dropping low-variance columns: (28362, 16)
Remaining columns: ['release_date', 'genre', 'len', 'violence', 'world/life', 'night/time', 'romantic', 'communication', 'obscene', 'music', 'movement/places', 'light/visual perceptions', 'sadness', 'feelings', 'topic', 'age']


---
## Step 3: Handle Missing Values

EDA confirmed zero null values across the entire dataset. We run the check again here on the cleaned dataframe as a verification step before proceeding.

In [4]:
# Verify no null values exist after column drops
nulls = df_clean.isnull().sum()
print('Missing values per column:')
print(
    nulls[nulls > 0]
    if nulls.sum() > 0
    else 'No missing values — no rows need to be dropped.'
)

# Drop any null rows as a safety measure
rows_before = len(df_clean)
df_clean = df_clean.dropna()
print(f'\nRows before: {rows_before:,}')
print(f'Rows after:  {len(df_clean):,}')

Missing values per column:
No missing values — no rows need to be dropped.

Rows before: 28,362
Rows after:  28,362


---
## Step 4: Remove Outliers Per Genre (Fixed Version)

**Why this approach matters:** The previous version applied IQR outlier removal globally across all songs. This caused every hip hop song to be removed because hip hop's legitimately high `obscene` scores looked like extreme outliers relative to all other genres combined.

**The fix:** We run outlier removal **within each genre separately**. This way, a hip hop song is only flagged as an outlier if it is extreme compared to *other hip hop songs* — not compared to country or pop songs. This preserves hip hop's real signal while still removing genuinely anomalous values within each genre.

We use a 3x IQR multiplier (wider than the standard 1.5x) since our data is already bounded between 0 and 1.

In [5]:
# Define the final set of score columns after all drops
# These are the features that will be fed into the KMeans model
score_cols_final = [
    'violence', 'world/life', 'night/time', 'romantic',
    'communication', 'obscene', 'music', 'movement/places',
    'light/visual perceptions', 'sadness', 'feelings'
]

# Keep only columns that exist in the dataframe
score_cols_final = [c for c in score_cols_final if c in df_clean.columns]
print(f'Score columns entering the model ({len(score_cols_final)} total):')
print(score_cols_final)

Score columns entering the model (11 total):
['violence', 'world/life', 'night/time', 'romantic', 'communication', 'obscene', 'music', 'movement/places', 'light/visual perceptions', 'sadness', 'feelings']


In [6]:
# Outlier removal within each genre separately
# This prevents minority genres like hip hop from being eliminated
# because their scores look extreme relative to majority genres

rows_before = len(df_clean)
genre_counts_before = df_clean['genre'].value_counts()

cleaned_genres = []

for genre in df_clean['genre'].unique():
    # Subset to just this genre
    genre_df = df_clean[df_clean['genre'] == genre].copy()

    # Apply IQR outlier removal within this genre only
    for col in score_cols_final:
        q1 = genre_df[col].quantile(0.25)
        q3 = genre_df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 3.0 * iqr
        upper = q3 + 3.0 * iqr
        genre_df = genre_df[
            (genre_df[col] >= lower) & (genre_df[col] <= upper)
        ]

    cleaned_genres.append(genre_df)

# Combine all genres back together
df_clean = pd.concat(cleaned_genres, axis=0)

rows_after = len(df_clean)
genre_counts_after = df_clean['genre'].value_counts()

print(f'Rows before outlier removal: {rows_before:,}')
print(f'Rows after outlier removal:  {rows_after:,}')
print(f'Rows removed: {rows_before - rows_after:,}')
print()
print('Songs per genre before vs after:')
comparison = pd.DataFrame({
    'before': genre_counts_before,
    'after': genre_counts_after
}).fillna(0).astype(int)
comparison['removed'] = comparison['before'] - comparison['after']
print(comparison)
print()
print('Hip hop songs remaining:', len(df_clean[df_clean['genre'] == 'hip hop']))

Rows before outlier removal: 28,362
Rows after outlier removal:  15,196
Rows removed: 13,166

Songs per genre before vs after:
         before  after  removed
genre                          
blues      4603   2541     2062
country    5444   2975     2469
hip hop     904    346      558
jazz       3844   2088     1756
pop        7038   3983     3055
reggae     2497   1318     1179
rock       4032   1945     2087

Hip hop songs remaining: 346


**Finding:** By running outlier removal within each genre separately, hip hop songs are now preserved in the dataset. The removal targets only songs that are extreme *within their own genre* — a hip hop song with a high obscene score is normal for hip hop and is no longer flagged as an outlier.

---
## Step 5: Scale Features with StandardScaler

KMeans calculates distance between every song and every cluster centroid. Without scaling, columns with larger spread dominate those distance calculations.

**EDA justification:** The variance analysis showed a 20x difference in spread between our strongest and weakest remaining features. `StandardScaler` transforms each column to mean=0 and standard deviation=1, putting all features on equal footing without changing the shape of any distribution.

**Important:** We save the fitted scaler so Notebook 3 can apply the exact same transformation to the recommendation dataset without re-fitting.

In [7]:
# Fit StandardScaler on training score columns only
# transform() re-centers each column to mean=0, std=1
scaler = StandardScaler()

X_scaled = pd.DataFrame(
    scaler.fit_transform(df_clean[score_cols_final]),
    columns=score_cols_final,
    index=df_clean.index
)

print('Before scaling — mean and std of first 3 columns:')
print(df_clean[score_cols_final[:3]].agg(['mean', 'std']).round(4))
print()
print('After scaling — should be ~0 mean and ~1 std:')
print(X_scaled[score_cols_final[:3]].agg(['mean', 'std']).round(4))

Before scaling — mean and std of first 3 columns:
      violence  world/life  night/time
mean    0.1382      0.1579      0.0350
std     0.1890      0.1901      0.0607

After scaling — should be ~0 mean and ~1 std:
      violence  world/life  night/time
mean       0.0         0.0         0.0
std        1.0         1.0         1.0


---
## Step 6: Build Final Dataframe and Save

We combine the label columns (`genre`, `topic`, `release_date`, `age`, `len`) with the scaled score columns. The label columns are retained so we can interpret cluster results in human terms after modeling — they are not fed into the clustering algorithm itself.

In [8]:
# Retain label columns for post-clustering interpretation
# These are NOT fed into KMeans — used to describe clusters afterward
label_cols = ['release_date', 'genre', 'topic', 'age', 'len']
label_cols = [c for c in label_cols if c in df_clean.columns]

# Combine label columns with scaled score columns
df_final = pd.concat(
    [
        df_clean[label_cols].reset_index(drop=True),
        X_scaled.reset_index(drop=True)
    ],
    axis=1
)

print(f'Final cleaned dataframe shape: {df_final.shape}')
print(f'Label columns retained: {label_cols}')
print(f'Score columns scaled:   {score_cols_final}')
print(f'\nGenre counts in final dataset:')
print(df_final['genre'].value_counts())
df_final.head()

Final cleaned dataframe shape: (15196, 16)
Label columns retained: ['release_date', 'genre', 'topic', 'age', 'len']
Score columns scaled:   ['violence', 'world/life', 'night/time', 'romantic', 'communication', 'obscene', 'music', 'movement/places', 'light/visual perceptions', 'sadness', 'feelings']

Genre counts in final dataset:
genre
pop        3983
country    2975
blues      2541
jazz       2088
rock       1945
reggae     1318
hip hop     346
Name: count, dtype: int64


,release_date,genre,topic,age,len,violence,world/life,night/time,romantic,communication,obscene,music,movement/places,light/visual perceptions,sadness,feelings
0,1950,pop,sadness,1.0,95,-0.393988,-0.827849,-0.567157,-0.139225,1.533405,-0.485793,-0.003149,-0.468710,-0.579193,1.012995,3.007352
1,1950,pop,world/life,1.0,51,-0.219207,1.502157,-0.555862,-0.495502,-0.756819,-0.481403,0.951672,-0.458008,2.345877,-0.855238,-0.545747
2,1950,pop,violence,1.0,98,1.494742,-0.825458,0.643342,-0.500706,-0.758836,-0.482882,-0.466773,-0.461615,1.787187,-0.229192,-0.552831
3,1950,pop,world/life,1.0,179,-0.728743,1.125333,0.093672,-0.513537,2.588802,-0.486530,-0.473682,-0.470509,0.024302,-0.859185,3.190521
4,1950,pop,sadness,1.0,61,-0.188669,-0.825104,0.311650,-0.499193,0.031014,-0.482452,-0.465958,1.172645,-0.571997,2.246317,-0.550771


In [9]:
# Save cleaned training data
OUTPUT_PATH = BASE_PATH + 'train_cleaned.csv'
df_final.to_csv(OUTPUT_PATH, index=False)
print(f'Saved cleaned data to: {OUTPUT_PATH}')
print(f'Shape: {df_final.shape}')

# Save scaler so Notebook 3 applies the same transformation to recommend.csv
SCALER_PATH = BASE_PATH + 'scaler.pkl'
with open(SCALER_PATH, 'wb') as f:
    pickle.dump(scaler, f)
print(f'Saved scaler to: {SCALER_PATH}')

# Save the cleaned index separately so Notebook 3 can align df_raw correctly
INDEX_PATH = BASE_PATH + 'cleaned_index.pkl'
with open(INDEX_PATH, 'wb') as f:
    pickle.dump(df_clean.index.tolist(), f)
print(f'Saved cleaned index to: {INDEX_PATH}')

Saved cleaned data to: /Users/saadult/Music Rec Algo/Music-Recommendation-Algorithm/data/train_cleaned.csv
Shape: (15196, 16)
Saved scaler to: /Users/saadult/Music Rec Algo/Music-Recommendation-Algorithm/data/scaler.pkl
Saved cleaned index to: /Users/saadult/Music Rec Algo/Music-Recommendation-Algorithm/data/cleaned_index.pkl


---
## Cleaning Summary

| Step | Action | Justification |
|---|---|---|
| Identity columns dropped | `Unnamed: 0`, `lyrics`, `artist_name`, `track_name` | Non-numeric, not usable by KMeans |
| Low-variance columns dropped | `shake the audience`, `family/gospel`, `family/spiritual`, `dating` | Variance < 0.003, confirmed in EDA |
| Null rows removed | 0 rows removed | EDA confirmed no missing values |
| Outliers removed | IQR per genre (3x multiplier) | Preserves hip hop — fixes global removal bug |
| Features scaled | StandardScaler (mean=0, std=1) | 20x variance difference requires equalization |

**Key fix:** Outlier removal now runs within each genre separately. This preserves hip hop songs whose high `obscene` scores are a genuine signal, not an anomaly.

**Output:** `data/train_cleaned.csv` — ready for KMeans modeling in Notebook 3.